In [ ]:
"""Script to analyze correlations between cyber attacks and their solutions from the dataset (to sense check graph.csv before passing it to the model)"""

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import networkx as nx
from scipy.stats import pearsonr
import warnings
warnings.filterwarnings('ignore')

# Set up plotting style
sns.set_palette("husl")

class CyberAttackSolutionAnalyzer:
    def __init__(self, csv_file_path):
        """
        Initialize the analyzer with the CSV file path
        
        """
        self.csv_file_path = csv_file_path
        self.data = None
        self.attack_columns = []
        self.solution_columns = []
        self.correlation_results = []
        
    def load_data(self):
        """
        Load and preprocess the CSV data
        
        """
        print("Loading data...")
        try:
            self.data = pd.read_csv(self.csv_file_path)
            print(f"Data loaded successfully. Shape: {self.data.shape}")
            
            # Identify attack and solution columns
            self._identify_columns()
            
            return True
        except Exception as e:
            print(f"Error loading data: {e}")
            return False
    
    def _identify_columns(self):
        """
        Identify attack and solution columns based on naming patterns
        
        """
        all_columns = self.data.columns.tolist()
        
        # Identify attack columns (ending with -ALL or starting with Mentions-)
        self.attack_columns = [col for col in all_columns 
                              if (col.endswith('-ALL') and not col.startswith('Solution_')) 
                              or col.startswith('Mentions-')]
        
        # Identify solution columns (starting with Solution_)
        self.solution_columns = [col for col in all_columns 
                                if col.startswith('Solution_')]
        
        print(f"Found {len(self.attack_columns)} attack columns")
        print(f"Found {len(self.solution_columns)} solution columns")
    
    def calculate_correlations(self, min_correlation=0.3):
        """
        Calculate correlations between attacks and solutions
        
        """
        print("Calculating correlations...")
        self.correlation_results = []
        
        for attack in self.attack_columns:
            if attack not in self.data.columns:
                continue
                
            attack_data = self.data[attack]
            
            for solution in self.solution_columns:
                if solution not in self.data.columns:
                    continue
                    
                solution_data = self.data[solution]
                
                # Remove rows with NaN values for this pair
                valid_mask = (~attack_data.isna()) & (~solution_data.isna())
                if valid_mask.sum() < 10:  # Need at least 10 data points
                    continue
                
                attack_clean = attack_data[valid_mask]
                solution_clean = solution_data[valid_mask]
                
                # Calculate Pearson correlation
                try:
                    correlation, p_value = pearsonr(attack_clean, solution_clean)
                    
                    if abs(correlation) >= min_correlation and p_value < 0.05:
                        self.correlation_results.append({
                            'Attack': attack,
                            'Solution': solution,
                            'Correlation': correlation,
                            'P_Value': p_value,
                            'Data_Points': len(attack_clean)
                        })
                except:
                    continue
        
        print(f"Found {len(self.correlation_results)} significant correlations")
        return self.correlation_results
    
    def generate_correlation_csv(self, output_file='attack_solution_correlations.csv'):
        """
        Generate CSV file with correlations 
        
        """
        if not self.correlation_results:
            print("No correlation results found. Run calculate_correlations() first.")
            return
        
        # Create a dictionary to store results in the required format
        results_dict = {}
        
        for result in self.correlation_results:
            attack = result['Attack']
            solution = result['Solution']
            correlation = result['Correlation']
            
            if attack not in results_dict:
                results_dict[attack] = {}
            
            results_dict[attack][solution] = correlation
        
        # Convert to DataFrame and save
        correlation_df = pd.DataFrame.from_dict(results_dict, orient='index')
        correlation_df = correlation_df.fillna(0)
        
        # Sort by strongest correlations
        correlation_df['max_corr'] = correlation_df.abs().max(axis=1)
        correlation_df = correlation_df.sort_values('max_corr', ascending=False)
        correlation_df = correlation_df.drop('max_corr', axis=1)
        
        # Save to CSV
        correlation_df.to_csv(output_file)
        print(f"Correlation CSV saved to {output_file}")
        
        return correlation_df
    
    def create_visualizations(self, top_n=15):
        """
        Create various visualizations for the correlation results
        
        """
        if not self.correlation_results:
            print("No correlation results found. Run calculate_correlations() first.")
            return
        
        # Convert to DataFrame for easier manipulation
        corr_df = pd.DataFrame(self.correlation_results)
        
        # Top Correlations Heatmap
        self._create_heatmap(corr_df, top_n)
        
        # Top Positive and Negative Correlations
        self._create_top_correlations_chart(corr_df, top_n)
        
        # Correlation Distribution
        self._create_correlation_distribution(corr_df)
        
        # Attack-Solution Network (if networkx is available)
        try:
            import networkx as nx
            self._create_network_visualization(corr_df, top_n)
        except ImportError:
            print("NetworkX not available. Skipping network visualization.")
        
        # Individual attack analysis for key attacks
        self._create_individual_attack_analysis(corr_df)
    
    def _create_heatmap(self, corr_df, top_n):
        """
        Create a heatmap of top correlations
        
        """
        # Get top correlations by absolute value
        top_corrs = corr_df.nlargest(top_n, 'Correlation')
        bottom_corrs = corr_df.nsmallest(top_n, 'Correlation')
        combined_corrs = pd.concat([top_corrs, bottom_corrs]).drop_duplicates()
        
        if len(combined_corrs) == 0:
            return
            
        # Create pivot table for heatmap
        pivot_data = combined_corrs.pivot_table(
            index='Attack', 
            columns='Solution', 
            values='Correlation',
            aggfunc='first'
        ).fillna(0)
        
        plt.figure(figsize=(16, 12))
        sns.heatmap(pivot_data, 
                   cmap='RdBu_r', 
                   center=0, 
                   annot=True, 
                   fmt='.2f',
                   cbar_kws={'label': 'Correlation Coefficient'})
        plt.title(f'Top {top_n} Cyber Attack - Solution Correlations\n(Heatmap)')
        plt.xticks(rotation=45, ha='right')
        plt.yticks(rotation=0)
        plt.tight_layout()
        plt.savefig('attack_solution_correlations_heatmap.png', dpi=300, bbox_inches='tight')
        plt.show()
    
    def _create_top_correlations_chart(self, corr_df, top_n):
        """
        Create bar chart of top positive and negative correlations
        
        """
        # Get top positive and negative correlations
        top_positive = corr_df.nlargest(top_n, 'Correlation')
        top_negative = corr_df.nsmallest(top_n, 'Correlation')
        
        fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 12))
        
        # Positive correlations
        if len(top_positive) > 0:
            y_pos = range(len(top_positive))
            bars1 = ax1.barh(y_pos, top_positive['Correlation'], color='green', alpha=0.7)
            ax1.set_yticks(y_pos)
            ax1.set_yticklabels([f"{row['Attack']}\nvs\n{row['Solution']}" 
                               for _, row in top_positive.iterrows()])
            ax1.set_xlabel('Correlation Coefficient')
            ax1.set_title(f'Top {top_n} Positive Correlations')
            ax1.grid(True, alpha=0.3)
            
            # Add value labels
            for i, bar in enumerate(bars1):
                width = bar.get_width()
                ax1.text(width + 0.01, bar.get_y() + bar.get_height()/2, 
                        f'{width:.3f}', ha='left', va='center')
        
        # Negative correlations
        if len(top_negative) > 0:
            y_pos = range(len(top_negative))
            bars2 = ax2.barh(y_pos, top_negative['Correlation'], color='red', alpha=0.7)
            ax2.set_yticks(y_pos)
            ax2.set_yticklabels([f"{row['Attack']}\nvs\n{row['Solution']}" 
                               for _, row in top_negative.iterrows()])
            ax2.set_xlabel('Correlation Coefficient')
            ax2.set_title(f'Top {top_n} Negative Correlations')
            ax2.grid(True, alpha=0.3)
            
            # Add value labels
            for i, bar in enumerate(bars2):
                width = bar.get_width()
                ax2.text(width - 0.01, bar.get_y() + bar.get_height()/2, 
                        f'{width:.3f}', ha='right', va='center')
        
        plt.tight_layout()
        plt.savefig('top_correlations_barchart.png', dpi=300, bbox_inches='tight')
        plt.show()
    
    def _create_correlation_distribution(self, corr_df):
        """
        Create distribution plot of correlation coefficients
        
        """
        plt.figure(figsize=(12, 6))
        
        plt.subplot(1, 2, 1)
        sns.histplot(corr_df['Correlation'], bins=30, kde=True)
        plt.axvline(x=0, color='red', linestyle='--', alpha=0.7)
        plt.xlabel('Correlation Coefficient')
        plt.ylabel('Frequency')
        plt.title('Distribution of Correlation Coefficients')
        
        plt.subplot(1, 2, 2)
        # Box plot by attack type
        if len(corr_df) > 0:
            # Get top attacks by number of correlations
            top_attacks = corr_df['Attack'].value_counts().head(10).index
            filtered_data = corr_df[corr_df['Attack'].isin(top_attacks)]
            
            if len(filtered_data) > 0:
                sns.boxplot(data=filtered_data, x='Correlation', y='Attack')
                plt.axvline(x=0, color='red', linestyle='--', alpha=0.7)
                plt.title('Correlation Distribution by Attack Type')
        
        plt.tight_layout()
        plt.savefig('correlation_distribution.png', dpi=300, bbox_inches='tight')
        plt.show()
    
    def _create_network_visualization(self, corr_df, top_n):
        """
        Create network visualization of attack-solution relationships
        
        """
        
        # Take top correlations
        top_corrs = corr_df.nlargest(top_n, abs(corr_df['Correlation']))
        
        G = nx.Graph()
        
        # Add nodes and edges
        for _, row in top_corrs.iterrows():
            attack = row['Attack'][:20]  # Truncate for display
            solution = row['Solution'][:20]
            correlation = row['Correlation']
            
            G.add_node(attack, type='attack', size=abs(correlation) * 1000)
            G.add_node(solution, type='solution', size=abs(correlation) * 1000)
            G.add_edge(attack, solution, weight=abs(correlation), 
                      color='red' if correlation < 0 else 'blue')
        
        plt.figure(figsize=(16, 12))
        pos = nx.spring_layout(G, k=1, iterations=50)
        
        # Draw nodes
        attack_nodes = [n for n in G.nodes() if G.nodes[n]['type'] == 'attack']
        solution_nodes = [n for n in G.nodes() if G.nodes[n]['type'] == 'solution']
        
        nx.draw_networkx_nodes(G, pos, nodelist=attack_nodes, 
                              node_color='lightcoral', 
                              node_size=[G.nodes[n]['size'] for n in attack_nodes],
                              alpha=0.8, label='Attacks')
        
        nx.draw_networkx_nodes(G, pos, nodelist=solution_nodes, 
                              node_color='lightblue', 
                              node_size=[G.nodes[n]['size'] for n in solution_nodes],
                              alpha=0.8, label='Solutions')
        
        # Draw edges
        edges = G.edges()
        colors = [G[u][v]['color'] for u, v in edges]
        weights = [G[u][v]['weight'] * 5 for u, v in edges]
        
        nx.draw_networkx_edges(G, pos, edge_color=colors, width=weights, alpha=0.6)
        
        # Draw labels
        nx.draw_networkx_labels(G, pos, font_size=8, font_weight='bold')
        
        plt.title(f'Cyber Attack - Solution Correlation Network\n(Top {top_n} correlations)')
        plt.axis('off')
        plt.legend()
        plt.tight_layout()
        plt.savefig('attack_solution_network.png', dpi=300, bbox_inches='tight')
        plt.show()
    
    def _create_individual_attack_analysis(self, corr_df):
        """
        Create detailed analysis for key attack types
        
        """
        key_attacks = [
            'DDoS-ALL', 'Phishing-ALL', 'Ransomware-ALL', 
            'Malware-ALL', 'Password Attack-ALL', 'Zero-day-ALL'
        ]
        
        available_attacks = [attack for attack in key_attacks 
                           if attack in corr_df['Attack'].unique()]
        
        if not available_attacks:
            return
        
        for attack in available_attacks[:3]:  # Limit to first 3 to avoid too many plots
            attack_data = corr_df[corr_df['Attack'] == attack].nlargest(10, 'Correlation')
            
            if len(attack_data) == 0:
                continue
                
            plt.figure(figsize=(12, 6))
            
            # Create horizontal bar chart
            y_pos = range(len(attack_data))
            colors = ['green' if x > 0 else 'red' for x in attack_data['Correlation']]
            
            plt.barh(y_pos, attack_data['Correlation'], color=colors, alpha=0.7)
            plt.yticks(y_pos, [sol.split('_')[-1].replace('_Mentions', '') 
                             for sol in attack_data['Solution']])
            plt.xlabel('Correlation Coefficient')
            plt.title(f'Top Solutions Correlated with {attack}')
            plt.grid(True, alpha=0.3)
            
            # Add value labels
            for i, v in enumerate(attack_data['Correlation']):
                plt.text(v + (0.01 if v > 0 else -0.01), i, f'{v:.3f}', 
                        ha='left' if v > 0 else 'right', va='center')
            
            plt.tight_layout()
            filename = f"{attack.replace('-ALL', '').lower()}_correlations.png"
            plt.savefig(filename, dpi=300, bbox_inches='tight')
            plt.show()

def main():
    """
    Main function to run the analysis
    
    """
    # Initialize analyzer
    csv_file_path = "./data/data.csv"  # Replace with your actual file path
    analyzer = CyberAttackSolutionAnalyzer(csv_file_path)
    
    # Load data
    if not analyzer.load_data():
        print("Failed to load data. Please check the file path.")
        return
    
    # Calculate correlations
    correlations = analyzer.calculate_correlations(min_correlation=0.3)
    
    if not correlations:
        print("No significant correlations found. Try lowering the min_correlation threshold.")
        return
    
    # Generate correlation CSV
    correlation_df = analyzer.generate_correlation_csv('attack_solution_correlations.csv')
    
    # Create visualizations
    print("Creating visualizations...")
    analyzer.create_visualizations(top_n=15)
    
    # Print summary statistics
    print("\n" + "="*50)
    print("SUMMARY STATISTICS")
    print("="*50)
    
    corr_df = pd.DataFrame(correlations)
    print(f"Total significant correlations: {len(corr_df)}")
    print(f"Average correlation: {corr_df['Correlation'].mean():.3f}")
    print(f"Median correlation: {corr_df['Correlation'].median():.3f}")
    print(f"Strongest positive correlation: {corr_df['Correlation'].max():.3f}")
    print(f"Strongest negative correlation: {corr_df['Correlation'].min():.3f}")
    
    # Top attacks by number of correlated solutions
    print("\nTop attacks by number of correlated solutions:")
    attack_counts = corr_df['Attack'].value_counts().head(10)
    for attack, count in attack_counts.items():
        print(f"  {attack}: {count} solutions")
    
    # Top solutions by number of correlated attacks
    print("\nTop solutions by number of correlated attacks:")
    solution_counts = corr_df['Solution'].value_counts().head(10)
    for solution, count in solution_counts.items():
        solution_name = solution.replace('Solution_', '').replace('_Mentions', '')
        print(f"  {solution_name}: {count} attacks")

if __name__ == "__main__":
    main()

In [ ]:
"""Script for feature importance and correlation analysis to determine which features impact cybersecurity threat levels the most."""

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestRegressor
from sklearn.feature_selection import SelectKBest, f_regression, mutual_info_regression
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from scipy.stats import spearmanr
import warnings
warnings.filterwarnings('ignore')

# Set style for better visualizations
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

def load_and_prepare_data(file_path):
    """
    Load the CSV file and prepare the data for analysis
    
    """
    print("Loading data...")
    df = pd.read_csv(file_path)
    
    # Set Month as index
    if 'Month' in df.columns:
        df.set_index('Month', inplace=True)
    
    print(f"Data shape: {df.shape}")
    print(f"Columns: {len(df.columns)}")
    
    return df

def handle_missing_data(df):
    """
    Handle missing values in the dataset
    
    """
    print("\nHandling missing data...")
    missing_percent = (df.isnull().sum() / len(df)) * 100
    print(f"Columns with missing values: {len(missing_percent[missing_percent > 0])}")
    
    # Remove columns with more than 50% missing values
    columns_to_drop = missing_percent[missing_percent > 50].index
    df_clean = df.drop(columns=columns_to_drop)
    
    # Fill remaining missing values with median
    df_clean = df_clean.fillna(df_clean.median())
    
    print(f"Data shape after cleaning: {df_clean.shape}")
    return df_clean

def quick_correlation_analysis(df, sample_size=100):
    """
    Quick correlation analysis using only highly variable features
    
    """
    print("\nPerforming quick correlation analysis...")
    
    # Select top N most variable features for correlation analysis
    feature_variance = df.var().sort_values(ascending=False)
    top_features = feature_variance.head(min(sample_size, len(feature_variance))).index
    
    print(f"Analyzing correlation for top {len(top_features)} most variable features...")
    
    # Calculate correlation matrix only for top features
    corr_matrix = df[top_features].corr()
    
    # Plot correlation heatmap
    plt.figure(figsize=(16, 14))
    mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
    sns.heatmap(corr_matrix, mask=mask, cmap='RdBu_r', center=0, 
                square=True, annot=False, cbar_kws={'shrink': 0.8})
    plt.title(f'Correlation Matrix of Top {len(top_features)} Most Variable Features', 
              fontsize=16, pad=20)
    plt.xticks(rotation=45, ha='right')
    plt.yticks(rotation=0)
    plt.tight_layout()
    plt.show()
    
    return corr_matrix

def random_forest_feature_importance(df, target_column=None, n_estimators=100, max_features='sqrt'):
    """
    Calculate feature importance using Random Forest (optimised for large datasets)
    
    """
    print("\nCalculating Random Forest feature importance...")
    
    # If no target specified, use the first column as example
    if target_column is None:
        # Try to find a suitable target (non-ALL column)
        potential_targets = [col for col in df.columns if 'ALL' in col and 'WAR/CONFLICT' not in col]
        if potential_targets:
            target_column = potential_targets[0]
            print(f"Using target: {target_column}")
        else:
            target_column = df.columns[0]
            print(f"No suitable target found, using: {target_column}")
    
    if target_column not in df.columns:
        target_column = df.columns[0]
    
    X = df.drop(columns=[target_column])
    y = df[target_column]
    
    # Use faster Random Forest with reduced parameters
    rf = RandomForestRegressor(
        n_estimators=n_estimators, 
        max_features=max_features,
        random_state=42, 
        n_jobs=-1,
        max_samples=0.8  # Use subsampling for speed
    )
    rf.fit(X, y)
    
    # Get feature importance
    feature_importance = pd.DataFrame({
        'feature': X.columns,
        'importance': rf.feature_importances_
    }).sort_values('importance', ascending=False)
    
    return feature_importance, target_column

def fast_statistical_feature_selection(df, target_column=None, top_k=100):
    """
    Fast statistical feature selection using mutual information
    
    """
    print("\nPerforming fast statistical feature selection...")
    
    if target_column is None:
        potential_targets = [col for col in df.columns if 'ALL' in col and 'WAR/CONFLICT' not in col]
        target_column = potential_targets[0] if potential_targets else df.columns[0]
    
    X = df.drop(columns=[target_column])
    y = df[target_column]
    
    # Use mutual information with limited features for speed
    mi_selector = SelectKBest(score_func=mutual_info_regression, k=top_k)
    mi_selector.fit(X, y)
    
    # Get scores for all features
    mi_scores = pd.DataFrame({
        'feature': X.columns,
        'mutual_info': mi_selector.scores_
    }).sort_values('mutual_info', ascending=False)
    
    return mi_scores

def pca_analysis(df, n_components=10):
    """
    Perform PCA to understand feature variance structure
    
    """
    print("\nPerforming PCA analysis...")
    
    # Standardize the data
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(df)
    
    # Perform PCA
    pca = PCA(n_components=n_components)
    pca_result = pca.fit_transform(X_scaled)
    
    return pca, pca_result

def variance_based_feature_selection(df, threshold=0.01):
    """
    Remove low-variance features
    
    """
    print("\nPerforming variance-based feature selection...")
    
    # Calculate variance for each feature
    variances = df.var()
    
    # Remove features with very low variance
    low_variance_features = variances[variances < threshold].index
    df_reduced = df.drop(columns=low_variance_features)
    
    print(f"Removed {len(low_variance_features)} low-variance features")
    print(f"Remaining features: {len(df_reduced.columns)}")
    
    return df_reduced

def visualize_feature_importance(results_dict, top_n=20):
    """
    Create comprehensive visualizations for feature importance results
    
    """
    print("\nCreating visualizations...")
    
    fig, axes = plt.subplots(2, 2, figsize=(20, 15))
    
    # Random Forest Feature Importance
    if 'rf_importance' in results_dict:
        rf_top = results_dict['rf_importance'].head(top_n)
        
        axes[0, 0].barh(range(len(rf_top)), rf_top['importance'])
        axes[0, 0].set_yticks(range(len(rf_top)))
        axes[0, 0].set_yticklabels(rf_top['feature'], fontsize=10)
        axes[0, 0].set_title(f'Top {top_n} Features - Random Forest Importance\n(Target: {results_dict["target_column"]})', fontsize=12)
        axes[0, 0].set_xlabel('Importance Score')
    
    # Mutual Information Scores
    if 'mi_scores' in results_dict:
        mi_top = results_dict['mi_scores'].head(top_n)
        
        axes[0, 1].barh(range(len(mi_top)), mi_top['mutual_info'], color='orange')
        axes[0, 1].set_yticks(range(len(mi_top)))
        axes[0, 1].set_yticklabels(mi_top['feature'], fontsize=10)
        axes[0, 1].set_title(f'Top {top_n} Features - Mutual Information', fontsize=12)
        axes[0, 1].set_xlabel('MI Score')
    
    # PCA Explained Variance
    if 'pca' in results_dict:
        pca = results_dict['pca']
        
        explained_variance = pca.explained_variance_ratio_ * 100
        cumulative_variance = np.cumsum(explained_variance)
        
        axes[1, 0].bar(range(1, len(explained_variance) + 1), explained_variance, 
                      alpha=0.6, color='skyblue', label='Individual')
        axes[1, 0].step(range(1, len(cumulative_variance) + 1), cumulative_variance, 
                       where='mid', label='Cumulative', color='red')
        axes[1, 0].set_xlabel('Principal Components')
        axes[1, 0].set_ylabel('Explained Variance (%)')
        axes[1, 0].set_title('PCA Explained Variance Ratio')
        axes[1, 0].legend()
        axes[1, 0].grid(True, alpha=0.3)
    
    # Feature Categories
    if 'rf_importance' in results_dict:
        rf_importance = results_dict['rf_importance']
        
        # Analyse features by category
        categories = {
            'DDoS': len([f for f in rf_importance['feature'] if 'DDoS' in f and 'ALL' not in f]),
            'Phishing': len([f for f in rf_importance['feature'] if 'Phishing' in f and 'ALL' not in f]),
            'Ransomware': len([f for f in rf_importance['feature'] if 'Ransomware' in f and 'ALL' not in f]),
            'Solutions': len([f for f in rf_importance['feature'] if 'Solution' in f]),
            'Dark Web': len([f for f in rf_importance['feature'] if 'DARK_WEB' in f]),
            'Vulnerability': len([f for f in rf_importance['feature'] if 'Vulnerability' in f]),
        }
        
        category_counts = {k: v for k, v in categories.items() if v > 0}
        
        axes[1, 1].pie(category_counts.values(), labels=category_counts.keys(), autopct='%1.1f%%')
        axes[1, 1].set_title('Feature Categories Distribution')
    
    plt.tight_layout()
    plt.show()

def feature_selection_summary(results_dict, top_n=30):
    """
    Generate a summary of the most important features across all methods
    
    """
    print("\n" + "="*80)
    print("FEATURE SELECTION SUMMARY")
    print("="*80)
    
    # Combine scores from different methods
    feature_scores = {}
    
    if 'rf_importance' in results_dict:
        for _, row in results_dict['rf_importance'].iterrows():
            feature_scores.setdefault(row['feature'], {})
            feature_scores[row['feature']]['rf_rank'] = _ + 1
            feature_scores[row['feature']]['rf_score'] = row['importance']
    
    if 'mi_scores' in results_dict:
        for _, row in results_dict['mi_scores'].iterrows():
            feature_scores.setdefault(row['feature'], {})
            feature_scores[row['feature']]['mi_rank'] = _ + 1
            feature_scores[row['feature']]['mi_score'] = row['mutual_info']
    
    # Calculate overall score (lower rank is better)
    overall_scores = []
    for feature, scores in feature_scores.items():
        overall_score = 0
        count = 0
        
        if 'rf_rank' in scores:
            overall_score += 1 / scores['rf_rank']
            count += 1
        if 'mi_rank' in scores:
            overall_score += 1 / scores['mi_rank']
            count += 1
        
        if count > 0:
            overall_scores.append((feature, overall_score / count))
    
    # Sort by overall score
    overall_scores.sort(key=lambda x: x[1], reverse=True)
    
    print(f"\nTop {top_n} Overall Most Important Features:")
    print("-" * 80)
    for i, (feature, score) in enumerate(overall_scores[:top_n], 1):
        print(f"{i:2d}. {feature}")
    
    return overall_scores

def export_feature_lists(results_dict, top_n=100):
    """
    Export feature lists for different selection criteria
    
    """
    print("\nExporting feature lists...")
    
    # Random Forest top features
    if 'rf_importance' in results_dict:
        rf_top = results_dict['rf_importance'].head(top_n)
        rf_top.to_csv('top_features_random_forest.csv', index=False)
        print(f"Saved top {len(rf_top)} Random Forest features")
    
    # Mutual Information top features
    if 'mi_scores' in results_dict:
        mi_top = results_dict['mi_scores'].head(top_n)
        mi_top.to_csv('top_features_mutual_information.csv', index=False)
        print(f"Saved top {len(mi_top)} Mutual Information features")
    
    # Combined important features
    if all(k in results_dict for k in ['rf_importance', 'mi_scores']):
        rf_features = set(results_dict['rf_importance'].head(top_n)['feature'])
        mi_features = set(results_dict['mi_scores'].head(top_n)['feature'])
        
        common_features = rf_features & mi_features
        pd.DataFrame({'feature': list(common_features)}).to_csv('common_important_features.csv', index=False)
        print(f"Saved {len(common_features)} common important features")

def main():
    """
    Main function to run the optimized feature importance analysis
    
    """
    # Replace with your actual file path
    file_path = "./data/data.csv" # Update this path
    
    try:
        # Load data
        df = load_and_prepare_data(file_path)
        
        # Handle missing data
        df_clean = handle_missing_data(df)
        
        # Store all results
        results = {}
        
        # Remove low-variance features first
        df_reduced = variance_based_feature_selection(df_clean, threshold=0.01)
        
        # Quick correlation analysis (limited to top features)
        quick_correlation_analysis(df_reduced, sample_size=50)
        
        # Random Forest Feature Importance (optimized)
        rf_importance, target_column = random_forest_feature_importance(
            df_reduced, 
            n_estimators=50,  
            max_features='sqrt'
        )
        results['rf_importance'] = rf_importance
        results['target_column'] = target_column
        
        # Statistical feature selection
        mi_scores = fast_statistical_feature_selection(df_reduced, target_column, top_k=100)
        results['mi_scores'] = mi_scores
        
        # PCA Analysis
        pca, pca_result = pca_analysis(df_reduced)
        results['pca'] = pca
        
        # Create visualizations
        visualize_feature_importance(results)
        
        # Generate summary
        overall_scores = feature_selection_summary(results)
        
        # Export results
        export_feature_lists(results)
        
        print("\n" + "="*80)
        print("ANALYSIS COMPLETE!")
        print("="*80)
        print(f"Original features: {df.shape[1]}")
        print(f"Features after cleaning: {df_clean.shape[1]}")
        print(f"Features after variance filtering: {df_reduced.shape[1]}")
        print(f"Target variable used: {target_column}")
        
        # Show top 10 features
        print(f"\nTop 10 Most Important Features:")
        for i, (feature, score) in enumerate(overall_scores[:10], 1):
            print(f"{i:2d}. {feature}")
        
    except FileNotFoundError:
        print(f"Error: File '{file_path}' not found.")
        print("Please update the 'file_path' variable with the correct path to your CSV file.")
    except Exception as e:
        print(f"An error occurred: {str(e)}")
        import traceback
        traceback.print_exc()

if __name__ == "__main__":
    main()

A script for Feature Importance analysis (to help with feature engineering decisions). 

In [ ]:
"""Script for establishing data-driven XAI analysis to determine data distribution and feature importance without model dependency, e.g. how robust is the data? what outliers are there? which features are most predictive based on data characteristics alone?"""

import os
import json
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.inspection import permutation_importance
from typing import Dict, List, Tuple, Optional, Any
import warnings
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE

warnings.filterwarnings('ignore')

class DirectAnalyzer:
    """ Direct XAI analysis on data """
    
    def __init__(self, data_path: str):
        self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        self.data_path = data_path
        self.data = None
        self.feature_names = None
        self.results = {}
        
    def load_and_analyze_data(self):
        """Load data and perform direct analysis """
        print("Loading data for direct XAI analysis...")
        
        try:
            # Load data with headers
            if self.data_path.endswith('.csv'):
                self.data = pd.read_csv(self.data_path)
            else:
                self.data = pd.read_csv(self.data_path, delim_whitespace=True)
            
            print(f"Data shape: {self.data.shape}")

            # Keep only numeric columns (drop any text-based identifiers)
            numeric_data = self.data.select_dtypes(include=[np.number])
            dropped = set(self.data.columns) - set(numeric_data.columns)
            if dropped:
                print(f"Non-numeric columns dropped: {', '.join(dropped)}")

            self.data = numeric_data

            # Use actual column headers as feature names
            self.feature_names = list(self.data.columns)

            # Normalize data
            self.scaler = StandardScaler()
            self.data_normalized = self.scaler.fit_transform(self.data)

            print("Data successfully loaded and normalized with real column headers.")
            return True
            
        except Exception as e:
            print(f"Error loading data: {e}")
            return False

            
        except Exception as e:
            print(f"Error loading data: {e}")
            return False
    
    def correlation_analysis(self):
        """Perform correlation-based feature importance"""
        print("Performing correlation analysis...")
        
        os.makedirs('results/correlation', exist_ok=True)
        
        try:
            # Calculate correlations between features
            corr_matrix = np.corrcoef(self.data_normalized.T)
            
            # Feature importance based on correlation with future values
            # (using time-shifted correlations as proxy for predictive power)
            importance_scores = []
            
            for i in range(self.data.shape[1]):
                # Correlation with future values (simple time series importance)
                if len(self.data) > 1:
                    # Use correlation with next time step as importance measure
                    corr_with_future = np.corrcoef(self.data_normalized[:-1, i], 
                                                  self.data_normalized[1:, i])[0, 1]
                    importance_scores.append(abs(corr_with_future))
                else:
                    # Fallback: variance as importance
                    importance_scores.append(np.var(self.data_normalized[:, i]))
            
            # Get top features
            top_k = min(15, len(importance_scores))
            top_indices = np.argsort(importance_scores)[-top_k:][::-1]
            top_features = [self.feature_names[i] for i in top_indices]
            top_scores = [importance_scores[i] for i in top_indices]
            
            # Plot feature importance
            plt.figure(figsize=(12, 8))
            plt.barh(range(len(top_scores)), top_scores)
            plt.yticks(range(len(top_scores)), top_features)
            plt.xlabel('Importance Score (Absolute Correlation)')
            plt.title('Feature Importance based on Temporal Correlation')
            plt.tight_layout()
            plt.savefig('results/correlation/feature_importance.png', dpi=300)
            plt.close()
            
            # Plot correlation matrix heatmap for top features
            plt.figure(figsize=(10, 8))
            top_corr_matrix = corr_matrix[top_indices][:, top_indices]
            sns.heatmap(top_corr_matrix, 
                       xticklabels=top_features,
                       yticklabels=top_features,
                       cmap='coolwarm', center=0,
                       annot=True, fmt='.2f')
            plt.title('Correlation Matrix of Top Features')
            plt.tight_layout()
            plt.savefig('results/correlation/correlation_heatmap.png', dpi=300)
            plt.close()
            
            result = {
                'top_features': top_features,
                'importance_scores': top_scores,
                'correlation_matrix': corr_matrix.tolist()
            }
            
            self.results['correlation'] = result
            print("Correlation analysis completed")
            return result
            
        except Exception as e:
            print(f"Correlation analysis failed: {e}")
            return None
    
    def statistical_analysis(self):
        """Perform statistical analysis of the data"""
        print("Performing statistical analysis...")
        
        os.makedirs('results/statistical', exist_ok=True)
        
        try:
            # Calculate various statistical measures
            statistics = {}
            
            # Basic statistics
            statistics['mean'] = np.mean(self.data_normalized, axis=0).tolist()
            statistics['std'] = np.std(self.data_normalized, axis=0).tolist()
            statistics['variance'] = np.var(self.data_normalized, axis=0).tolist()
            
            # Temporal statistics (change over time)
            if len(self.data_normalized) > 1:
                differences = np.diff(self.data_normalized, axis=0)
                statistics['avg_change'] = np.mean(np.abs(differences), axis=0).tolist()
                statistics['volatility'] = np.std(differences, axis=0).tolist()
            
            # Identify outliers using IQR method
            Q1 = np.percentile(self.data_normalized, 25, axis=0)
            Q3 = np.percentile(self.data_normalized, 75, axis=0)
            IQR = Q3 - Q1
            outlier_mask = (self.data_normalized < (Q1 - 1.5 * IQR)) | (self.data_normalized > (Q3 + 1.5 * IQR))
            statistics['outlier_percentage'] = (np.sum(outlier_mask, axis=0) / len(self.data_normalized) * 100).tolist()
            
            # Plot statistical distributions
            num_features_to_plot = min(6, self.data.shape[1])
            fig, axes = plt.subplots(2, 3, figsize=(15, 10))
            axes = axes.ravel()
            
            for i in range(num_features_to_plot):
                axes[i].hist(self.data_normalized[:, i], bins=20, alpha=0.7, edgecolor='black')
                axes[i].set_title(f'{self.feature_names[i]} Distribution')
                axes[i].set_xlabel('Value')
                axes[i].set_ylabel('Frequency')
            
            for i in range(num_features_to_plot, 6):
                fig.delaxes(axes[i])
            
            plt.tight_layout()
            plt.savefig('results/statistical/feature_distributions.png', dpi=300)
            plt.close()
            
            # Plot outlier analysis
            plt.figure(figsize=(12, 6))
            features_to_show = min(10, len(statistics['outlier_percentage']))
            indices = np.argsort(statistics['outlier_percentage'])[-features_to_show:][::-1]
            outlier_percentages = [statistics['outlier_percentage'][i] for i in indices]
            feature_names = [self.feature_names[i] for i in indices]
            
            plt.bar(range(len(outlier_percentages)), outlier_percentages)
            plt.xticks(range(len(outlier_percentages)), feature_names, rotation=45)
            plt.xlabel('Features')
            plt.ylabel('Outlier Percentage (%)')
            plt.title('Percentage of Outliers by Feature')
            plt.tight_layout()
            plt.savefig('results/statistical/outlier_analysis.png', dpi=300)
            plt.close()
            
            self.results['statistical'] = statistics
            print("Statistical analysis completed")
            return statistics
            
        except Exception as e:
            print(f"Statistical analysis failed: {e}")
            return None
    
    def time_series_analysis(self):
        """Analyze time series patterns in the data"""
        print("Performing time series analysis...")
        
        os.makedirs('results/time_series', exist_ok=True)
        
        try:
            # Analyze temporal patterns
            time_series_stats = {}
            
            # Autocorrelation analysis
            autocorr_lags = min(10, len(self.data_normalized) // 4)
            autocorrelations = []
            
            for feature_idx in range(min(10, self.data.shape[1])):
                feature_data = self.data_normalized[:, feature_idx]
                autocorr = []
                for lag in range(1, autocorr_lags + 1):
                    if len(feature_data) > lag:
                        corr = np.corrcoef(feature_data[:-lag], feature_data[lag:])[0, 1]
                        autocorr.append(corr)
                    else:
                        autocorr.append(0)
                autocorrelations.append(autocorr)
            
            time_series_stats['autocorrelations'] = autocorrelations
            
            # Plot autocorrelations for top features
            num_features_to_plot = min(5, len(autocorrelations))
            plt.figure(figsize=(12, 8))
            
            for i in range(num_features_to_plot):
                plt.plot(range(1, autocorr_lags + 1), autocorrelations[i], 
                        marker='o', label=f'Feature {i}')
            
            plt.axhline(y=0, color='r', linestyle='--', alpha=0.5)
            plt.xlabel('Lag')
            plt.ylabel('Autocorrelation')
            plt.title('Autocorrelation of Top Features')
            plt.legend()
            plt.grid(True, alpha=0.3)
            plt.tight_layout()
            plt.savefig('results/time_series/autocorrelation.png', dpi=300)
            plt.close()
            
            # Plot time series of most important features (based on variance)
            variances = np.var(self.data_normalized, axis=0)
            top_var_indices = np.argsort(variances)[-6:][::-1]
            
            fig, axes = plt.subplots(2, 3, figsize=(15, 10))
            axes = axes.ravel()
            
            for i, feature_idx in enumerate(top_var_indices):
                axes[i].plot(self.data_normalized[:, feature_idx])
                axes[i].set_title(f'{self.feature_names[feature_idx]} (var: {variances[feature_idx]:.3f})')
                axes[i].set_xlabel('Time')
                axes[i].set_ylabel('Value')
                axes[i].grid(True, alpha=0.3)
            
            plt.tight_layout()
            plt.savefig('results/time_series/feature_trajectories.png', dpi=300)
            plt.close()
            
            # Trend analysis
            if len(self.data_normalized) > 5:
                trends = []
                for feature_idx in range(min(10, self.data.shape[1])):
                    x = np.arange(len(self.data_normalized))
                    y = self.data_normalized[:, feature_idx]
                    z = np.polyfit(x, y, 1)
                    trends.append(z[0])  # Slope
                
                time_series_stats['trends'] = trends
                
                # Plot trends
                plt.figure(figsize=(10, 6))
                feature_indices = range(min(10, len(trends)))
                plt.bar(feature_indices, [trends[i] for i in feature_indices])
                plt.xlabel('Feature Index')
                plt.ylabel('Trend Slope')
                plt.title('Linear Trend of Features Over Time')
                plt.xticks(feature_indices, [f'F{i}' for i in feature_indices])
                plt.tight_layout()
                plt.savefig('results/time_series/trend_analysis.png', dpi=300)
                plt.close()
            
            self.results['time_series'] = time_series_stats
            print(" Time series analysis completed")
            return time_series_stats
            
        except Exception as e:
            print(f" Time series analysis failed: {e}")
            return None
    
    def clustering_analysis(self):
        """Perform clustering to identify patterns in the data"""
        print("Performing clustering analysis...")
        
        os.makedirs('results/clustering', exist_ok=True)
        
        try:
            
            
            # Use PCA for dimensionality reduction
            pca = PCA(n_components=2)
            data_2d = pca.fit_transform(self.data_normalized)
            
            # Perform clustering
            n_clusters = min(5, len(self.data_normalized) // 10)
            if n_clusters < 2:
                n_clusters = 2
                
            kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
            clusters = kmeans.fit_predict(self.data_normalized)
            
            # Plot clusters in 2D space
            plt.figure(figsize=(10, 8))
            scatter = plt.scatter(data_2d[:, 0], data_2d[:, 1], c=clusters, cmap='viridis', alpha=0.7)
            plt.colorbar(scatter, label='Cluster')
            plt.xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.2%} variance)')
            plt.ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.2%} variance)')
            plt.title('Data Clustering in 2D PCA Space')
            plt.tight_layout()
            plt.savefig('results/clustering/pca_clusters.png', dpi=300)
            plt.close()
            
            # Analyze cluster characteristics
            cluster_stats = {}
            for cluster_id in range(n_clusters):
                cluster_data = self.data_normalized[clusters == cluster_id]
                if len(cluster_data) > 0:
                    cluster_stats[cluster_id] = {
                        'size': len(cluster_data),
                        'mean_features': np.mean(cluster_data, axis=0).tolist(),
                        'std_features': np.std(cluster_data, axis=0).tolist()
                    }
            
            # Plot cluster sizes
            plt.figure(figsize=(8, 6))
            cluster_sizes = [stats['size'] for stats in cluster_stats.values()]
            plt.bar(range(len(cluster_sizes)), cluster_sizes)
            plt.xlabel('Cluster ID')
            plt.ylabel('Number of Points')
            plt.title('Cluster Sizes')
            plt.xticks(range(len(cluster_sizes)))
            plt.tight_layout()
            plt.savefig('results/clustering/cluster_sizes.png', dpi=300)
            plt.close()
            
            # Feature importance based on cluster separation
            feature_importance = []
            for feature_idx in range(self.data.shape[1]):
                # Calculate between-cluster variance / within-cluster variance
                overall_mean = np.mean(self.data_normalized[:, feature_idx])
                between_var = 0
                within_var = 0
                
                for cluster_id, stats in cluster_stats.items():
                    cluster_mean = stats['mean_features'][feature_idx]
                    cluster_size = stats['size']
                    between_var += cluster_size * (cluster_mean - overall_mean) ** 2
                    
                    cluster_data = self.data_normalized[clusters == cluster_id, feature_idx]
                    within_var += np.sum((cluster_data - cluster_mean) ** 2)
                
                if within_var > 0:
                    importance = between_var / within_var
                else:
                    importance = 0
                
                feature_importance.append(importance)
            
            # Plot clustering-based feature importance
            top_k = min(15, len(feature_importance))
            top_indices = np.argsort(feature_importance)[-top_k:][::-1]
            top_features = [self.feature_names[i] for i in top_indices]
            top_scores = [feature_importance[i] for i in top_indices]
            
            plt.figure(figsize=(12, 6))
            plt.barh(range(len(top_scores)), top_scores)
            plt.yticks(range(len(top_scores)), top_features)
            plt.xlabel('Cluster Separation Importance')
            plt.title('Feature Importance based on Cluster Separation')
            plt.tight_layout()
            plt.savefig('results/clustering/clustering_importance.png', dpi=300)
            plt.close()
            
            result = {
                'clusters': cluster_stats,
                'feature_importance': feature_importance,
                'pca_variance_ratio': pca.explained_variance_ratio_.tolist()
            }
            
            self.results['clustering'] = result
            print("Clustering analysis completed")
            return result
            
        except Exception as e:
            print(f"Clustering analysis failed: {e}")
            return None
    
    def generate_comprehensive_report(self):
        """Generate comprehensive XAI report based on data analysis"""
        print("Generating comprehensive report...")
        
        os.makedirs('results', exist_ok=True)
        
        # Create summary report
        report = {
            'data_info': {
                'shape': self.data.shape,
                'num_features': self.data.shape[1],
                'num_samples': self.data.shape[0]
            },
            'analysis_results': {}
        }
        
        # Summarize each analysis
        for method, result in self.results.items():
            if method == 'correlation' and 'top_features' in result:
                report['analysis_results'][method] = {
                    'top_5_features': result['top_features'][:5],
                    'status': 'completed'
                }
            elif method == 'statistical':
                report['analysis_results'][method] = {
                    'features_analyzed': len(result.get('mean', [])),
                    'status': 'completed'
                }
            elif method == 'time_series':
                report['analysis_results'][method] = {
                    'autocorrelation_lags': len(result.get('autocorrelations', [[]])[0]) if result.get('autocorrelations') else 0,
                    'status': 'completed'
                }
            elif method == 'clustering':
                report['analysis_results'][method] = {
                    'num_clusters': len(result.get('clusters', {})),
                    'status': 'completed'
                }
        
        # Save JSON report
        with open('results/comprehensive_analysis_report.json', 'w') as f:
            json.dump(report, f, indent=2)
        
        # Create executive summary
        with open('results/executive_summary.txt', 'w') as f:
            f.write("DATA-DRIVEN XAI ANALYSIS REPORT\n")
            f.write("=" * 50 + "\n\n")
            
            f.write("EXECUTIVE SUMMARY\n")
            f.write("-" * 20 + "\n")
            f.write(f"Dataset: {self.data_path}\n")
            f.write(f"Shape: {self.data.shape[0]} samples × {self.data.shape[1]} features\n")
            f.write(f"Analyses completed: {len(self.results)}\n\n")
            
            f.write("KEY FINDINGS:\n")
            f.write("-" * 15 + "\n")
            
            if 'correlation' in self.results:
                top_features = self.results['correlation'].get('top_features', [])[:3]
                f.write(f"• Top predictive features: {', '.join(top_features)}\n")
            
            if 'statistical' in self.results:
                outlier_pct = self.results['statistical'].get('outlier_percentage', [])
                if outlier_pct:
                    max_outliers = max(outlier_pct)
                    f.write(f"• Maximum outlier percentage: {max_outliers:.1f}%\n")
            
            if 'clustering' in self.results:
                num_clusters = len(self.results['clustering'].get('clusters', {}))
                f.write(f"• Natural data clusters identified: {num_clusters}\n")
            
            f.write("\nRECOMMENDATIONS:\n")
            f.write("-" * 15 + "\n")
            f.write("1. Focus on top correlated features for threat prediction\n")
            f.write("2. Monitor high-volatility features for anomaly detection\n")
            f.write("3. Use clustering patterns to identify threat categories\n")
            f.write("4. Consider temporal patterns for proactive threat detection\n")
        
        print(" Comprehensive report generated")
    
    def run_complete_analysis(self):
        """Run complete data-driven XAI analysis"""
        print("Starting data-driven XAI analysis...")
        print("=" * 60)
        
        # Load data
        if not self.load_and_analyze_data():
            print(" Failed to load data")
            return False
        
        print("\n" + "=" * 60)
        print("RUNNING DATA ANALYSIS METHODS")
        print("=" * 60)
        
        # Run analyses
        methods = [
            ("Correlation Analysis", self.correlation_analysis),
            ("Statistical Analysis", self.statistical_analysis),
            ("Time Series Analysis", self.time_series_analysis),
            ("Clustering Analysis", self.clustering_analysis),
        ]
        
        successful_methods = 0
        for method_name, method_func in methods:
            print(f"\n--- {method_name} ---")
            try:
                result = method_func()
                if result is not None:
                    print(f" SUCCESS: {method_name}")
                    successful_methods += 1
                else:
                    print(f"  PARTIAL: {method_name}")
            except Exception as e:
                print(f" FAILED: {method_name} - {e}")
        
        # Generate report
        print(f"\n--- Generating Report ---")
        self.generate_comprehensive_report()
        
        print("\n" + "=" * 60)
        print("ANALYSIS COMPLETE")
        print("=" * 60)
        print(f"Successful analyses: {successful_methods}/{len(methods)}")
        print(f"Results saved in: results/")
        
        return successful_methods > 0

def main():
    """Main function"""
    
    # Update this path to your data file
    data_path = './data/sm_data_g.csv'
    
    # Try alternative paths
    alternative_paths = [
        './data/sm_data_g.csv',
        '../data/sm_data_g.csv',
        './sm_data_g.csv',
        'sm_data_g.csv'
    ]
    
    working_path = None
    for path in alternative_paths:
        if os.path.exists(path):
            working_path = path
            print(f"Found data file: {path}")
            break
    
    if working_path is None:
        print("Could not find data file. Please check the path.")
        return
    
    # Run analysis
    analyzer = DirectAnalyzer(working_path)
    success = analyzer.run_complete_analysis()
    
    if success:
        print("\n Data-driven XAI analysis completed successfully!")
        print("Check the 'results/' directory for all outputs")
    else:
        print("\n Analysis completed with some limitations.")
        print("Basic results are available in 'results/'")

if __name__ == "__main__":
    main()